# 02 — Baselines: Random Forest and Gradient Boosting

The first models the spec asks for (section I): *"Random Forest /
Gradient Boosting on hand-crafted pulse-transit-time and PPG-morphology
features — classical, interpretable, fast baseline widely used in this
exact literature"*.

No GPU needed — this reads `data/features/*.parquet`, 61 numbers per
clip instead of the waveform. It runs on a laptop or on Colab.

**On Colab, cap the random forest.** The free tier gives two CPU cores,
and 300 trees over 465,480 clips will crawl. `run(model='rf', ..., cap=100)`
caps it at 100 clips per training subject. Gradient boosting is fine
either way. On a laptop, run both uncapped.

This is also the cheapest sanity check in the project: if the
calibration-free MAE lands near the 12-13 mmHg the published PulseDB
papers report, the whole pipeline — data, features, splits, metrics — is
confirmed in ten minutes rather than after an hour and a half of GPU time.

Each model is scored under all four protocols. The model, the features
and the training clips are identical across the four; the only thing
that changes is which patients are in the test set.

| protocol | who is tested | what it tells you |
|---|---|---|
| `leaky` | same patients, random clips | the number much of the field reports |
| `calbased` | same patients, official held-out clips | a device that already knows you |
| `calfree` | 144 unseen patients | **the honest number** |
| `aami` | 116 unseen patients | the zero-shot end of the personalization curve |

Roughly 10 minutes for gradient boosting, 40 for the random forest.

## Setup

Run these cells in order before anything else. They work both on Colab
and on a laptop — on Colab the data is copied off Drive onto the local
disk first, because Drive is a network mount and the sampler reads clips
in random order, which would leave the GPU waiting on I/O. The copy also
stops immediately if the upload turned out to be incomplete.

In [ ]:
import os, sys, glob, shutil, time

IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content/drive')

if not IN_COLAB:
    # running from notebooks/ on a normal machine
    sys.path.insert(0, '../src')
    sys.path.insert(0, '../scripts')
else:
    from google.colab import drive
    drive.mount('/content/drive')

    # Find the bp folder wherever it was dropped in Drive.
    hits = glob.glob('/content/drive/MyDrive/**/data/colab_manifest.json',
                     recursive=True)
    if len(hits) != 1:
        raise SystemExit(f'expected exactly one bp folder, found {len(hits)}:'
                         + ''.join('\n  ' + h for h in hits))
    DRIVE_ROOT = os.path.dirname(os.path.dirname(hits[0]))
    LOCAL_ROOT = '/content/bp'
    print('drive:', DRIVE_ROOT)

    # Copy the data to local disk first. Drive is a network mount and the
    # sampler reads clips in random order, so training straight off Drive
    # leaves the GPU idle waiting on I/O. A few minutes here saves hours.
    t0 = time.time()
    for rel in ['src', 'scripts', 'data/colab_manifest.json', 'data/features',
                'data/processed/pulsedb', 'data/processed/ppg_dalia/windows']:
        src, dst = f'{DRIVE_ROOT}/{rel}', f'{LOCAL_ROOT}/{rel}'
        if os.path.exists(dst):
            continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        (shutil.copytree if os.path.isdir(src) else shutil.copy2)(src, dst)
        print(f'  copied {rel:40s} {time.time() - t0:5.0f}s')

    # results/ points back at Drive — metrics, predictions and checkpoints
    # all land under results/models/<name>/, so a dropped session does not
    # take the run with it.
    os.makedirs(f'{DRIVE_ROOT}/results/models', exist_ok=True)
    if not os.path.islink(f'{LOCAL_ROOT}/results'):
        os.symlink(f'{DRIVE_ROOT}/results', f'{LOCAL_ROOT}/results')

    os.environ['BP_ROOT'] = LOCAL_ROOT     # every path follows this
    sys.path.insert(0, LOCAL_ROOT + '/src')
    sys.path.insert(0, LOCAL_ROOT + '/scripts')

    # Fail now if the upload was incomplete, not twenty minutes into a run.
    from prepare_colab_upload import verify
    if verify(LOCAL_ROOT):
        raise SystemExit('copy is incomplete — see the list above')
    print(f'\nBP_ROOT = {LOCAL_ROOT}   (ready in {time.time() - t0:.0f}s)')

In [ ]:
from train import run, PROTOCOLS
print(PROTOCOLS)

### Gradient boosting, all four protocols

In [ ]:
gb = {p: run(model='gb', protocol=p) for p in PROTOCOLS}

### Random forest, all four protocols

Slower (300 trees). Pass `cap=100` for a quick pass — at most 100 clips
per training subject, about five minutes.

In [ ]:
rf = {p: run(model='rf', protocol=p) for p in PROTOCOLS}

### The comparability run

Balancing is on by default because spec section L asks for it, and it lowers the error on the hypertensive patients considerably. It also raises the headline MAE, and every published PulseDB number was measured without it — so one extra run per model gives the figure that lines up against the literature.

In [ ]:
nb = {m: run(model=m, protocol='calfree', balance=False, tag='nobalance')
      for m in ('gb', 'rf')}

### The leakage gap in one table

This is the project's actual output: one model, one training set, four
answers, differing only in who was tested.

In [ ]:
import pandas as pd

rows = []
for name, res in (('gb', gb), ('rf', rf)):
    res = dict(res, **{'calfree (no bal.)': nb[name]})
    for proto, out in res.items():
        for target in ('sbp', 'dbp'):
            m = out['results'][target]
            rows.append(dict(model=name, protocol=proto, target=target,
                             mae=m['mae'], sde=m['sde'], r=m['r'],
                             aami='PASS' if m['aami_pass'] else 'FAIL',
                             bhs=m['bhs']))

df = pd.DataFrame(rows).round(2)
display(df.pivot_table(index=['model', 'target'], columns='protocol',
                       values='mae').round(2))
df

### Hyperparameters

Spec section L: *"k-fold (subject-level) within the training subjects for
hyperparameter tuning"*. Folds are formed over subjects, never over clips,
so a candidate is never rewarded for memorising a patient. None of the
test subsets are touched.

In [ ]:
from train import tune

grid = [dict(learning_rate=lr, max_leaf_nodes=n)
        for lr in (0.03, 0.06, 0.12) for n in (31, 63)]
best = tune(model='gb', grid=grid, folds=5, cap=100)